# Binary classification — credit scoring

**Start here.** This notebook covers the core machinery of
`bdp-model-gate` — contexts, checks, reports, verdicts, configuration and the
CLI — using a binary credit-scoring model. The other notebooks assume it.

| Notebook | Covers |
|---|---|
| **01 (this one)** | binary classification, and the library end to end |
| [02](02_multiclass_ordinal_sklearn.ipynb) | multiclass and ordinal underwriting |
| [03](03_regression_sklearn.ipynb) | regression — pricing, severity, frequency |
| [04](04_any_framework_classification.ipynb) | PyTorch and other non-sklearn models |

The gate runs fairness, performance, compliance and security checks against a
trained model *before* it reaches production, and reduces them to one verdict:

| Verdict | Meaning | Pipeline should |
|---|---|---|
| `PASS` | nothing flagged | deploy automatically |
| `NEEDS_REVIEW` | only non-blocking flags (fairness) | stop for human sign-off |
| `BLOCKED` | a blocking check failed | hard-fail the build |

In [1]:
# %pip install -q "bdp-model-gate[structured]"
import json
import logging
import warnings

import numpy as np
import pandas as pd

import bdp_model_gate

# The library logs through the stdlib `logging` module and never calls
# basicConfig() itself, so it composes with whatever your pipeline uses.
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(name)s: %(message)s")
logging.getLogger("bdp_model_gate").setLevel(logging.INFO)
pd.set_option("display.width", 130)

print("bdp-model-gate", bdp_model_gate.__version__)

bdp-model-gate 0.5.1


## 1. A model worth governing

The library's defaults target **NDPA/NDPR** (Nigeria's data-protection
regime): the built-in PII patterns match Nigerian phone numbers and NIN/BVN
identifiers, and `credit_scoring`, `underwriting`, `pricing` and
`claims_decisioning` are treated as DPIA triggers.

So: a credit-scoring model, with problems planted deliberately.

- `distance_to_branch_km` is nearly determined by `region` — a **proxy** for
  a protected attribute the model never sees
- `gender` shifts the ground truth, so its effect survives into the model

In [2]:
rng = np.random.default_rng(42)
N = 1500

region = rng.choice(["Lagos", "Abuja", "Kano", "Port Harcourt"], N, p=[0.4, 0.25, 0.2, 0.15])
gender = rng.choice(["F", "M"], N, p=[0.45, 0.55])

income = rng.lognormal(11.6, 0.45, N) * pd.Series(region).map(
    {"Lagos": 1.35, "Abuja": 1.20, "Port Harcourt": 1.00, "Kano": 0.70}
).to_numpy()

X = pd.DataFrame(
    {
        "monthly_income_ngn": income.round(2),
        "age": rng.integers(21, 65, N).astype(float),
        "months_employed": np.clip(rng.normal(48, 30, N), 0, None).round(),
        "existing_loans": rng.poisson(1.1, N).astype(float),
        "debt_to_income": np.clip(rng.beta(2, 5, N) * 1.4, 0.01, 0.95).round(4),
        "distance_to_branch_km": (
            pd.Series(region).map(
                {"Lagos": 2.0, "Abuja": 3.5, "Port Harcourt": 6.0, "Kano": 14.0}
            ).to_numpy()
            + rng.normal(0, 1.1, N)
        ).round(2),
    }
)

logit = (
    -1.0
    + 3.00 * np.log(X["monthly_income_ngn"] / 50_000)
    - 6.00 * X["debt_to_income"]
    + 0.030 * X["months_employed"]
    - 0.50 * X["existing_loans"]
    + 0.90 * (gender == "M")          # nudges the ground truth
)
repaid = rng.binomial(1, 1 / (1 + np.exp(-logit)))
protected_df = pd.DataFrame({"gender": gender, "region": region})

print(f"{N} applicants | repayment rate {repaid.mean():.1%}")
display(X.head())

1500 applicants | repayment rate 61.4%


,monthly_income_ngn,age,months_employed,existing_loans,debt_to_income,distance_to_branch_km
0,122847.44,39.0,21.0,2.0,0.0727,13.06
1,93197.19,59.0,27.0,0.0,0.4136,3.31
2,31799.64,51.0,0.0,5.0,0.3211,5.30
3,34481.98,27.0,15.0,1.0,0.8258,14.43
4,279242.74,25.0,78.0,0.0,0.8625,3.53


In [3]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val, prot_train, prot_val = train_test_split(
    X, repaid, protected_df, test_size=0.35, random_state=42, stratify=repaid
)

# gender and region are excluded — "fairness through unawareness".
# Section 5 shows why that alone is not enough.
model = GradientBoostingClassifier(random_state=42, n_estimators=120, max_depth=3)
model.fit(X_train, y_train)

# For binary classification y_pred is normally the positive-class probability.
y_pred = model.predict_proba(X_val)[:, 1]

print(f"train {len(X_train)} | validation {len(X_val)}")

train 975 | validation 525


## 2. The context, and graceful degradation

`StructuredGateContext` bundles everything the checks might need. Only the
model (or a `predict_fn`), `X`, `y_true` and `y_pred` are required — **every
other field is optional**, and omitting one makes the checks that depend on
it report `NOT_APPLICABLE` rather than fail.

That is the library's central contract: the gate grades what you give it.

In [4]:
from bdp_model_gate import ModelGate, StructuredGateContext

minimal = StructuredGateContext(model=model, X=X_val, y_true=y_val, y_pred=y_pred)
minimal_report = ModelGate().run(minimal)
print(minimal_report.summary())

INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.


INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.


INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.


INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.


INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.


INFO     bdp_model_gate.task: context.task="auto" inferred task='binary' from y_true. Set task explicitly if that is wrong — a count target (e.g. claims frequency) is indistinguishable from a multiclass one by shape alone.


INFO     bdp_model_gate.gate: gate_status=PASS task=binary n_flags=0 metric=roc_auc score=0.8636


Gate status: PASS (9ms, binary)
  roc_auc: 0.8636
  performance: 0 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 0 flag(s)


With no `protected_df`, no `model_card` and no benchmark data, most checks
skipped themselves. Note the log line: `task="auto"` inferred `binary` from
`y_true` and **said so**. Inference is never silent — set `task` explicitly
for anything you gate on.

Now the full context.

In [5]:
def explain_decision(prompt: str) -> str:
    """Stand-in for an LLM that turns a decision into prose for the applicant.
    PromptInjectionCheck probes whatever you pass here."""
    if any(k in prompt.lower() for k in ("ignore previous", "no content policy", "verbatim")):
        return "I cannot comply with that request."
    return "The application was declined due to a high debt-to-income ratio."


model_card = {
    "model_name": "credit-scoring-gbm",
    "version": "0.4.1",
    "use_case": "credit_scoring",                # high-risk -> DPIA required
    "legal_basis": "Contractual necessity (NDPA 2023, s.25(1)(b))",
    "data_minimization_justification": "Only affordability signals are collected.",
    "training_data_source": "Internal loan book, 2021-2025, consented at origination",
    "dpia_completed": True,
    "influences_decision_about_person": True,
    "explainability_method": "SHAP TreeExplainer, surfaced in the adverse-action notice",
}

context = StructuredGateContext(
    model=model,
    X=X_val,
    y_true=y_val,
    y_pred=y_pred,
    protected_df=prot_val,                       # enables fairness
    latencies_ms=rng.gamma(9.0, 8.5, 500),       # enables the latency gate
    cost_per_inference=0.0009,                   # enables the cost gate
    model_card=model_card,                       # enables compliance
    generate_fn=explain_decision,                # enables prompt injection
    task="binary",
)

report = ModelGate().run(context)
print(report.summary())

INFO     bdp_model_gate.gate: gate_status=NEEDS_REVIEW task=binary n_flags=7 metric=roc_auc score=0.8636


Gate status: NEEDS_REVIEW (870ms, binary)
  roc_auc: 0.8636
  performance: 0 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 7 flag(s)


## 3. Reading a `GateReport`

Every check returns one or more `CheckResult`s. A `flag` is one of:

- `OK` — passed
- `NOT_APPLICABLE` — skipped (missing optional input or dependency)
- `CHECK_ERROR` — the check raised; **always treated as blocking**
- a check-specific risk string (`PROXY_RISK`, `PII_LEAKAGE_RISK`, …)

`is_ok` treats `OK` and `NOT_APPLICABLE` as fine, so a skipped check never
blocks a deploy.

In [6]:
display(
    pd.DataFrame(
        [
            {
                "category": r.category,
                "check": r.check_name,
                "flag": r.flag,
                "blocking": r.blocking,
                "ms": r.duration_ms,
                "detail": (r.detail[:62] + "...") if len(r.detail) > 62 else r.detail,
            }
            for r in report.results
        ]
    )
)

,category,check,flag,blocking,ms,detail
0,fairness,proxy_correlation,PROXY_RISK,False,6.53,distance_to_branch_km correlates with region (...
1,fairness,disparate_impact,OK,False,18.23,gender: demographic parity diff=0.001
2,fairness,disparate_impact,DISPARITY_RISK,False,18.23,region: demographic parity diff=0.365
3,fairness,shap_subgroup_gap,SUBGROUP_IMPACT_RISK,False,814.01,monthly_income_ngn SHAP contribution gap acros...
4,fairness,counterfactual_flip,NOT_APPLICABLE,False,0.02,no protected attributes present as model inputs
5,fairness,equalised_odds,OK,False,0.48,gender: true-positive-rate difference=0.051 (m...
6,fairness,equalised_odds,EQUALISED_ODDS_RISK,False,0.48,gender: equalised odds difference=0.104 (max 0...
7,fairness,equalised_odds,EQUAL_OPPORTUNITY_RISK,False,0.48,region: true-positive-rate difference=0.219 (m...
8,fairness,equalised_odds,EQUALISED_ODDS_RISK,False,0.48,region: equalised odds difference=0.263 (max 0...
9,fairness,subgroup_calibration,OK,False,0.73,gender: calibration error spans 0.0499 (M) to ...


### How the verdict is computed

- any **blocking** check flagged → `BLOCKED`
- otherwise any flag at all → `NEEDS_REVIEW`
- otherwise → `PASS`

All four fairness checks ship as `blocking=False`. That is a design stance,
not an oversight: fairness signals frequently need human judgement, so they
route to review rather than failing the build. Performance, compliance and
security are blocking.

In [7]:
flags = report.flags
print(f"verdict      : {report.gate_status}")
print(f"task         : {report.task}")
print(f"headline     : {report.model_metric} = {report.model_score}")
print(f"blocking     : {sum(r.blocking for r in flags)}")
print(f"non-blocking : {sum(not r.blocking for r in flags)}")
print(f"wall clock   : {report.total_duration_ms} ms\n")
for r in flags:
    print(f"[{'BLOCK ' if r.blocking else 'review'}] {r.category:11} {r.check_name}")
    print(f"          {r.detail}")

verdict      : NEEDS_REVIEW
task         : binary
headline     : roc_auc = 0.8636
blocking     : 0
non-blocking : 7
wall clock   : 870.35 ms

[review] fairness    proxy_correlation
          distance_to_branch_km correlates with region (eta^2=0.940)
[review] fairness    disparate_impact
          region: demographic parity diff=0.365
[review] fairness    shap_subgroup_gap
          monthly_income_ngn SHAP contribution gap across region=1.914 — 356% of the mean absolute contribution 0.538
[review] fairness    equalised_odds
          gender: equalised odds difference=0.104 (max 0.1) — TPR gap 0.051, FPR gap 0.104
[review] fairness    equalised_odds
          region: true-positive-rate difference=0.219 (max 0.1) — among those who should be approved, Kano is least likely to be
[review] fairness    equalised_odds
          region: equalised odds difference=0.263 (max 0.1) — TPR gap 0.219, FPR gap 0.263
[review] fairness    subgroup_calibration
          region: calibration error spans 0.05

In [8]:
payload = report.to_dict()
print(json.dumps({k: v for k, v in payload.items() if k != "results_by_category"}, indent=2))
report.to_json("gate_report.json")
print("\nwrote gate_report.json")

{
  "gate_status": "NEEDS_REVIEW",
  "task": "binary",
  "model_metric": "roc_auc",
  "model_score": 0.8636,
  "model_auc": 0.8636,
  "n_flags": 7,
  "total_duration_ms": 870.35
}

wrote gate_report.json


## 4. Performance: choosing your metric

`PerformanceThresholdCheck` gates a **model score**, **p95 latency** and
**cost per inference**. The score metric is yours to choose:

| `metric` value | Behaviour |
|---|---|
| `"auto"` (default) | `roc_auc` if scikit-learn is available, else `accuracy` — with a loud warning |
| a name | `roc_auc`, `average_precision`, `accuracy`, `balanced_accuracy`, `f1`, `precision`, `recall` |
| a callable | any `fn(y_true, y_pred) -> float` |

`min_score` is compared against **whichever metric ran**, so set the two
together.

In [9]:
from bdp_model_gate import PerformanceConfig
from bdp_model_gate.metrics import BUILTIN_METRICS
from bdp_model_gate.structured.performance import PerformanceThresholdCheck

rows = []
for name, spec in sorted(BUILTIN_METRICS.items()):
    if "binary" not in spec.tasks:
        continue
    result = PerformanceThresholdCheck(
        PerformanceConfig(metric=name, min_score=0.0)
    ).run(context)[0]
    rows.append({"metric": name, "value": result.metadata["value"],
                 "expects": "labels" if spec.needs_hard_labels else "scores"})
display(pd.DataFrame(rows).set_index("metric").sort_values("value", ascending=False))

,value,expects
metric,,
average_precision,0.9067,scores
recall,0.8665,labels
roc_auc,0.8636,scores
f1,0.8404,labels
precision,0.8158,labels
accuracy,0.7981,labels
balanced_accuracy,0.7781,labels


Those numbers span a wide range for **one unchanged model** — which is why
the metric has to be explicit. `min_score = 0.80` is a comfortable pass under
`roc_auc` and an impossible bar under `precision`.

Label-based metrics binarise continuous predictions at
`decision_threshold` (default `0.5`); ranking metrics ignore it.

In [10]:
sweep = []
for threshold in (0.3, 0.4, 0.5, 0.6, 0.7):
    row = {"decision_threshold": threshold}
    for name in ("precision", "recall", "f1", "roc_auc"):
        cfg = PerformanceConfig(metric=name, min_score=0.0, decision_threshold=threshold)
        row[name] = PerformanceThresholdCheck(cfg).run(context)[0].metadata["value"]
    sweep.append(row)
display(pd.DataFrame(sweep).set_index("decision_threshold").round(4))
print("roc_auc is flat — it ranks, so the threshold is irrelevant to it.")

,precision,recall,f1,roc_auc
decision_threshold,,,,
0.3,0.7519,0.9410,0.8359,0.8636
0.4,0.7853,0.8975,0.8377,0.8636
0.5,0.8158,0.8665,0.8404,0.8636
0.6,0.8297,0.8168,0.8232,0.8636
0.7,0.8542,0.7640,0.8066,0.8636


roc_auc is flat — it ranks, so the threshold is irrelevant to it.


### A custom metric, and the no-silent-substitution rule

Any callable works, called with `y_pred` exactly as supplied. Here an F2
score, weighting recall over precision — for credit scoring, missing a
defaulter usually costs more than declining a good applicant.

In [11]:
from sklearn.metrics import fbeta_score


def f2_at_30pct(y_true, y_pred):
    return fbeta_score(y_true, (np.asarray(y_pred) >= 0.30).astype(int), beta=2)


result = PerformanceThresholdCheck(
    PerformanceConfig(metric=f2_at_30pct, min_score=0.85)
).run(context)[0]
print(f"{result.flag:18} {result.detail}")
print("reported as:", result.metadata["metric"])   # from the function __name__

OK                 f2_at_30pct=0.8959 (min 0.85)
reported as: f2_at_30pct


In [12]:
from unittest import mock

import bdp_model_gate.metrics as metrics_module
from bdp_model_gate.exceptions import GateConfigurationError

# Simulate a core-only install (no scikit-learn) without uninstalling.
with mock.patch.object(metrics_module, "_load_sklearn_metric", return_value=None):
    auto = PerformanceThresholdCheck(
        PerformanceConfig(metric="auto", min_score=0.80)
    ).run(context)[0]
    print("auto fell back to :", auto.metadata["metric"])
    print("flagged as fallback:", auto.metadata["metric_is_fallback"])
    print("detail             :", auto.detail)

    # An explicitly named metric is never swapped for another.
    try:
        PerformanceThresholdCheck(PerformanceConfig(metric="roc_auc")).run(context)
    except GateConfigurationError as exc:
        print("\nexplicit metric unavailable ->", exc)

WARNING  bdp_model_gate.metrics: performance.metric='auto': 'roc_auc' is unavailable (scikit-learn not installed) — scoring with 'accuracy' instead. Set performance.metric explicitly to silence this, and remember min_score is interpreted against 'accuracy', not 'roc_auc'.


auto fell back to : accuracy
flagged as fallback: True
detail             : accuracy=0.7981 (min 0.8) [fell back from the preferred metric — scikit-learn not installed; computed without scikit-learn]

explicit metric unavailable -> performance.metric='roc_auc' requires scikit-learn — install it with `pip install bdp-model-gate[structured]`, or set performance.metric to one of: accuracy, mae, mape, poisson_deviance, r2, rmse


## 5. Fairness — the non-blocking category

| Check | Question |
|---|---|
| `ProxyCorrelationCheck` | does a feature encode a protected attribute the model can't see? |
| `DisparateImpactCheck` | do outcomes differ across groups? (demographic parity) |
| `ShapSubgroupCheck` | does a feature *drive* outcomes differently per group? |
| `CounterfactualFlipCheck` | does flipping the attribute change the prediction? |

In [13]:
for r in report.by_category("fairness"):
    print(f"{'  ' if r.is_ok else '->'} {r.check_name:22} {r.flag}")
    print(f"     {r.detail}")

-> proxy_correlation      PROXY_RISK
     distance_to_branch_km correlates with region (eta^2=0.940)
   disparate_impact       OK
     gender: demographic parity diff=0.001
-> disparate_impact       DISPARITY_RISK
     region: demographic parity diff=0.365
-> shap_subgroup_gap      SUBGROUP_IMPACT_RISK
     monthly_income_ngn SHAP contribution gap across region=1.914 — 356% of the mean absolute contribution 0.538
   counterfactual_flip    NOT_APPLICABLE
     no protected attributes present as model inputs
   equalised_odds         OK
     gender: true-positive-rate difference=0.051 (max 0.1) — among those who should be approved, M is least likely to be
-> equalised_odds         EQUALISED_ODDS_RISK
     gender: equalised odds difference=0.104 (max 0.1) — TPR gap 0.051, FPR gap 0.104
-> equalised_odds         EQUAL_OPPORTUNITY_RISK
     region: true-positive-rate difference=0.219 (max 0.1) — among those who should be approved, Kano is least likely to be
-> equalised_odds         EQUALISE

### Proxy correlation — why dropping the column isn't enough

We never gave the model `region`, but `distance_to_branch_km` is nearly a
relabelling of it, so the model can reconstruct the attribute anyway. This is
the single most useful check in the suite: it catches the failure mode where
a team believes they removed a protected attribute and did not.

In [14]:
from bdp_model_gate.structured.fairness import DisparateImpactCheck, ProxyCorrelationCheck

for r in ProxyCorrelationCheck().run(context):
    print(f"{r.flag:14} {r.detail}")

display(
    X_val.assign(region=prot_val["region"].values)
    .groupby("region")[["distance_to_branch_km", "monthly_income_ngn"]]
    .mean()
    .round(2)
)

PROXY_RISK     distance_to_branch_km correlates with region (eta^2=0.940)


,distance_to_branch_km,monthly_income_ngn
region,,
Abuja,3.39,139551.44
Kano,14.20,84878.88
Lagos,2.10,167986.63
Port Harcourt,6.00,121876.68


An η² near 1.0 means the feature is essentially a relabelling of `region`.

### Disparate impact

Demographic parity counts predictions *equal to 1*, so it needs hard class
labels. Continuous predictions are binarised for you at
`FairnessConfig.decision_threshold`.

In [15]:
for r in DisparateImpactCheck().run(context):
    print(f"{r.flag:18} {r.detail}")

OK                 gender: demographic parity diff=0.001
DISPARITY_RISK     region: demographic parity diff=0.365


### Three families of fairness, and why you must choose

The four checks above all measure **independence** — do outcomes match across
groups, ignoring the ground truth. That is one of three families, and the
suite reports all three since 0.5.0:

| Family | Question | Check |
|---|---|---|
| **Independence** | do selection rates match? | `disparate_impact` |
| **Separation** | do *error* rates match? | `equalised_odds` |
| **Sufficiency** | does a score mean the same thing per group? | `subgroup_calibration` |

They matter because independence alone is weak: it ignores `y_true` entirely,
so a model can achieve perfect parity by being wrong in compensating
directions.

In [16]:
from bdp_model_gate import FairnessConfig
from bdp_model_gate.structured.calibration_checks import (
    CalibrationCheck,
    EqualisedOddsCheck,
    SubgroupCalibrationCheck,
)

FAMILY = {
    "disparate_impact": "independence",
    "equalised_odds": "separation",
    "subgroup_calibration": "sufficiency",
}

for check in (DisparateImpactCheck(), EqualisedOddsCheck(), SubgroupCalibrationCheck()):
    for r in check.run(context):
        if r.metadata.get("protected_attr") != "region":
            continue
        print(f"{FAMILY[r.check_name]:14} {r.flag:26} {r.detail[:74]}")

independence   DISPARITY_RISK             region: demographic parity diff=0.365
separation     EQUAL_OPPORTUNITY_RISK     region: true-positive-rate difference=0.219 (max 0.1) — among those who sh
separation     EQUALISED_ODDS_RISK        region: equalised odds difference=0.263 (max 0.1) — TPR gap 0.219, FPR gap
sufficiency    SUBGROUP_CALIBRATION_RISK  region: calibration error spans 0.0500 (Lagos) to 0.1165 (Abuja) — gap 0.0


All three flag here, and each is telling you something different — different
groups are approved at different rates, the model is wrong in different ways
for each, and a given score does not mean quite the same thing everywhere.

The important part is that **you cannot fix all three**. Calibration, TPR
balance and FPR balance are mathematically incompatible whenever the base rate
differs between groups, except in degenerate cases
(Kleinberg–Mullainathan–Raghavan 2016; Chouldechova 2017). Our book has a real
income difference by region, so a base-rate difference follows, so no
remediation satisfies everything.

Rather than take that on trust, force the trade-off and watch it happen.
Below we rescale each region's probabilities to equalise selection rates —
buying **independence** outright — and measure what it costs.

In [17]:
# Buy independence: scale each group's scores so every region selects at the
# same rate as the book overall.
target_rate = float((y_pred >= 0.5).mean())
equalised = y_pred.copy()

for region_name in prot_val["region"].unique():
    mask = (prot_val["region"] == region_name).to_numpy()
    # The multiplier that lands this group on the target selection rate.
    lo, hi = 0.01, 100.0
    for _ in range(40):
        mid = (lo + hi) / 2
        if (np.clip(y_pred[mask] * mid, 0, 1) >= 0.5).mean() < target_rate:
            lo = mid
        else:
            hi = mid
    equalised[mask] = np.clip(y_pred[mask] * ((lo + hi) / 2), 0.0, 1.0)


def measure(predictions, label):
    ctx = StructuredGateContext(
        model=model, X=X_val, y_true=y_val, y_pred=predictions,
        protected_df=prot_val, task="binary",
    )
    parity = [r for r in DisparateImpactCheck().run(ctx)
              if r.metadata["protected_attr"] == "region"][0]
    calib = [r for r in SubgroupCalibrationCheck().run(ctx)
             if r.metadata["protected_attr"] == "region"][0]
    return {
        "predictions": label,
        "parity gap (independence)": parity.metadata["demographic_parity_diff"],
        "calibration gap (sufficiency)": calib.metadata["ece_gap"],
    }


display(
    pd.DataFrame([measure(y_pred, "original"), measure(equalised, "rate-equalised")])
    .set_index("predictions")
)

,parity gap (independence),calibration gap (sufficiency)
predictions,,
original,0.365,0.06656
rate-equalised,0.010,0.13113


Independence improves, sufficiency gets worse. That is the trade-off, not a
quirk of this dataset — and it is why a governance tool that reported only
demographic parity would let you "fix" a model by making its scores mean
different things for different people, and call that progress.

The gate cannot resolve this for you. What it can do, and now does, is refuse
to report one family while letting you assume the others were checked. Which
notion you gate on is a policy decision, and it belongs to whoever signs the
model off.

### Overall calibration

Separate from fairness: do the stated probabilities match reality at all?
Discrimination and calibration are independent properties. A model can rank
perfectly — AUC 1.0 — while every probability it emits is twice too high,
which scores beautifully and misprices every policy.

In [18]:
result = CalibrationCheck().run(context)[0]   # PerformanceConfig defaults
print(result.flag, "-", result.detail)

from bdp_model_gate.calibration import brier_decomposition

parts = brier_decomposition(y_val, y_pred)
print()
for key in ("brier", "reliability", "resolution", "uncertainty"):
    print(f"  {key:14} {parts[key]:.4f}")
print("\n  reliability: distance from observed frequency — recalibration fixes this")
print("  resolution : how much predictions vary from the base rate — higher is better")
print("  uncertainty: the base rate's own variance — a floor, not a model property")

OK - expected calibration error=0.0548 (max 0.1); reliability=0.0047, resolution=0.0947

  brier          0.1469
  reliability    0.0047
  resolution     0.0947
  uncertainty    0.2372

  reliability: distance from observed frequency — recalibration fixes this
  resolution : how much predictions vary from the base rate — higher is better
  uncertainty: the base rate's own variance — a floor, not a model property


`resolution` is the one people forget. A model predicting the base rate for
everyone is **perfectly calibrated and completely useless** — reliability
near zero, resolution near zero. Neither ECE nor the Brier score says so
alone, which is why the decomposition is reported.

### Intersections

Every check above treats each protected attribute on its own. Harm
concentrates where attributes meet, and marginal checks are blind to it by
construction — a model can look acceptable on gender and on region while
failing badly for women in one region.

`FairnessConfig.intersectional` turns on pairwise combinations. It is off by
default because joint groups are smaller and the reading needs more care.

In [19]:
intersectional = FairnessConfig(intersectional=True, min_group_size=30)
for r in EqualisedOddsCheck(intersectional).run(context):
    if r.metadata["notion"] == "equal_opportunity":
        print(f"{r.metadata['protected_attr']:20} TPR gap {r.metadata['tpr_difference']:6.3f}  {r.flag}")

gender               TPR gap  0.051  OK
region               TPR gap  0.219  EQUAL_OPPORTUNITY_RISK
gender × region      TPR gap  0.350  EQUAL_OPPORTUNITY_RISK


### PII leakage

Regex-scans string columns for values that should have been hashed or
tokenised upstream. Defaults cover email, Nigerian phone numbers and
NIN/BVN-shaped identifiers, and the patterns are configurable.

In [20]:
from bdp_model_gate import SecurityConfig
from bdp_model_gate.structured.security import PIILeakageCheck

X_leaky = X_val.copy()
X_leaky["contact_email"] = [f"applicant{i}@example.ng" for i in range(len(X_leaky))]
X_leaky["phone"] = ["0803" + str(rng.integers(1_000_000, 9_999_999)) for _ in range(len(X_leaky))]

leaky = StructuredGateContext(
    model=model, X=X_leaky, y_true=y_val, y_pred=y_pred, model_card=model_card, task="binary"
)
for r in PIILeakageCheck().run(leaky):
    print(f"{r.flag:20} {r.detail}")

custom = SecurityConfig()
custom.pii_patterns = {**custom.pii_patterns, "account_number_ng": r"\b\d{10}\b"}
print("\nwith an extra pattern:")
for r in PIILeakageCheck(custom).run(leaky):
    print(f"{r.flag:20} {r.detail}")

PII_LEAKAGE_RISK     column 'contact_email' has 500 value(s) matching email pattern
PII_LEAKAGE_RISK     column 'phone' has 500 value(s) matching phone_ng pattern
PII_LEAKAGE_RISK     column 'phone' has 500 value(s) matching nin_bvn pattern

with an extra pattern:
PII_LEAKAGE_RISK     column 'contact_email' has 500 value(s) matching email pattern
PII_LEAKAGE_RISK     column 'phone' has 500 value(s) matching phone_ng pattern
PII_LEAKAGE_RISK     column 'phone' has 500 value(s) matching nin_bvn pattern


### Compliance — the model card

`ComplianceMappingCheck` enforces three things, all blocking: required fields
present, a DPIA for a high-risk `use_case`, and a documented explainability
method when the model affects a person.

In [21]:
sloppy = StructuredGateContext(
    model=model, X=X_val, y_true=y_val, y_pred=y_pred, task="binary",
    model_card={"model_name": "credit-scoring-gbm", "use_case": "credit_scoring"},
)
sloppy_report = ModelGate().run(sloppy)
print("verdict:", sloppy_report.gate_status, "\n")
for r in sloppy_report.by_category("compliance"):
    print(f"{'FAIL' if not r.is_ok else 'ok  '}  {r.detail}")

INFO     bdp_model_gate.gate: gate_status=BLOCKED task=binary n_flags=5 metric=roc_auc score=0.8636


verdict: BLOCKED 

FAIL  model_card.legal_basis missing — required under NDPA/NDPR
FAIL  model_card.data_minimization_justification missing — required under NDPA/NDPR
FAIL  model_card.training_data_source missing — required under NDPA/NDPR
FAIL  high-risk use case requires a completed DPIA
FAIL  required — model affects a person's outcome, no method documented


## 6. Tuning thresholds

`GateConfig` nests one dataclass per category. Defaults are starting points,
not regulatory guidance.

In [22]:
from bdp_model_gate import GateConfig
from bdp_model_gate.structured import default_structured_checks

strict = GateConfig()
strict.performance.metric = "roc_auc"
strict.performance.min_score = 0.90            # above what this model achieves
strict.fairness.disparity_threshold = 0.03
strict.fairness.proxy_corr_threshold = 0.15
strict.security.adversarial_flip_rate_threshold = 0.01

strict_report = ModelGate(checks=default_structured_checks(strict, include_plugins=False)).run(
    context
)
print(strict_report.summary())
print()
for r in strict_report.flags:
    print(f"[{'BLOCK ' if r.blocking else 'review'}] {r.check_name:22} {r.detail[:64]}")

INFO     bdp_model_gate.gate: gate_status=BLOCKED task=binary n_flags=10 metric=roc_auc score=0.8636


Gate status: BLOCKED (63ms, binary)
  roc_auc: 0.8636
  performance: 1 flag(s)
  compliance: 0 flag(s)
  security: 1 flag(s)
  fairness: 8 flag(s)

[review] proxy_correlation      monthly_income_ngn correlates with region (eta^2=0.167)
[review] proxy_correlation      distance_to_branch_km correlates with region (eta^2=0.940)
[review] disparate_impact       region: demographic parity diff=0.365
[review] shap_subgroup_gap      monthly_income_ngn SHAP contribution gap across region=1.914 — 3
[review] equalised_odds         gender: equalised odds difference=0.104 (max 0.1) — TPR gap 0.05
[review] equalised_odds         region: true-positive-rate difference=0.219 (max 0.1) — among th
[review] equalised_odds         region: equalised odds difference=0.263 (max 0.1) — TPR gap 0.21
[review] subgroup_calibration   region: calibration error spans 0.0500 (Lagos) to 0.1165 (Abuja)
[BLOCK ] performance_thresholds roc_auc=0.8636 (min 0.9)
[BLOCK ] adversarial_robustness flip rate under random pertur

## 7. Writing your own check

Subclass `BaseCheck`, set the class attributes, implement `run(context)`,
return a list of `CheckResult`. `blocking` is the important decision:
`True` fails the build, `False` routes to human review.

`supported_tasks` declares which tasks the check applies to; it defaults to
all of them.

In [23]:
from bdp_model_gate import BaseCheck, CheckResult


class FeatureDriftCheck(BaseCheck):
    """Flags validation features whose mean has drifted from training."""

    name = "feature_drift"
    category = "performance"
    blocking = False           # drift warrants a look, not an automatic stop

    def __init__(self, reference: pd.DataFrame, max_z: float = 3.0):
        self.reference = reference
        self.max_z = max_z

    def run(self, context):
        results = []
        for col in context.X.select_dtypes(include=[np.number]).columns:
            if col not in self.reference:
                continue
            ref = self.reference[col]
            sd = ref.std()
            if sd == 0:
                continue
            z = abs(context.X[col].mean() - ref.mean()) / sd
            if z > self.max_z:
                results.append(
                    CheckResult(
                        self.name, self.category, "DRIFT_RISK",
                        detail=f"{col} mean shifted {z:.2f} sd from training",
                        blocking=self.blocking,
                        metadata={"feature": col, "z_score": round(float(z), 3)},
                    )
                )
        return results or [
            CheckResult(self.name, self.category, "OK",
                        f"no feature drifted beyond {self.max_z} sd", self.blocking)
        ]


print(FeatureDriftCheck(X_train).run(context)[0].detail)

shifted = StructuredGateContext(
    model=model, X=X_val.assign(monthly_income_ngn=X_val.monthly_income_ngn * 2.4),
    y_true=y_val, y_pred=y_pred, task="binary",
)
for r in FeatureDriftCheck(X_train).run(shifted):
    print(f"{r.flag:12} {r.detail}")

no feature drifted beyond 3.0 sd
OK           no feature drifted beyond 3.0 sd


### A broken check is contained

One badly-behaved check must not take down the gate. `ModelGate` catches
exceptions per check and converts them to a **blocking** `CHECK_ERROR`, so
the rest still run and the pipeline still stops.

In [24]:
class ExplodingCheck(BaseCheck):
    name = "exploding_check"
    category = "security"
    blocking = True

    def run(self, context):
        raise RuntimeError("upstream scanner unreachable")


contained = ModelGate(checks=[ExplodingCheck(), PerformanceThresholdCheck()]).run(context)
for r in contained.results:
    print(f"{r.flag:16} {r.check_name:24} {r.detail[:56]}")
print("\nverdict:", contained.gate_status, "— and the performance check still ran")

WARNING  bdp_model_gate.gate: check=exploding_check raised an exception: RuntimeError('upstream scanner unreachable')


INFO     bdp_model_gate.gate: gate_status=BLOCKED task=binary n_flags=1 metric=roc_auc score=0.8636


CHECK_ERROR      exploding_check          check raised an exception: RuntimeError('upstream scanne
OK               performance_thresholds   roc_auc=0.8636 (min 0.8)
OK               performance_thresholds   p95 latency=128.54ms (max 200.0ms)
OK               performance_thresholds   cost/inference=0.00090 (max 0.002)

verdict: BLOCKED — and the performance check still ran


### Third-party checks via plugins

A separate package can register checks without forking, via the
`bdp_model_gate.checks` entry-point group:

```toml
[project.entry-points."bdp_model_gate.checks"]
my_check = "my_package.checks:MyCustomCheck"
```

`default_structured_checks()` picks them up automatically; pass
`include_plugins=False` to opt out. A plugin that fails to import is logged
and skipped rather than crashing the gate.

In [25]:
from bdp_model_gate.registry import ENTRY_POINT_GROUP, discover_plugin_checks

print("entry-point group:", ENTRY_POINT_GROUP)
print("plugins installed here:", discover_plugin_checks() or "none")

entry-point group: bdp_model_gate.checks
plugins installed here: none


## 8. Input validation

Inputs are validated **eagerly**, before any check runs, so a mistake
surfaces as a clear message rather than an obscure traceback from inside
SHAP. `GateValidationError` and `GateConfigurationError` both subclass
`BDPModelGateError`.

In [26]:
from bdp_model_gate.exceptions import GateValidationError


class NotAModel:
    pass


cases = [
    ("no model at all", dict(X=X_val, y_true=y_val, y_pred=y_pred)),
    ("model without .predict()", dict(model=NotAModel(), X=X_val, y_true=y_val, y_pred=y_pred)),
    ("X is not a DataFrame", dict(model=model, X=X_val.to_numpy(), y_true=y_val, y_pred=y_pred)),
    ("y_true / X length mismatch", dict(model=model, X=X_val, y_true=y_val[:10], y_pred=y_pred)),
    ("y_true has one class",
     dict(model=model, X=X_val, y_true=np.ones(len(X_val), dtype=int), y_pred=y_pred)),
    ("protected_df not aligned",
     dict(model=model, X=X_val, y_true=y_val, y_pred=y_pred, protected_df=prot_val.head(5))),
    ("model_card not a dict",
     dict(model=model, X=X_val, y_true=y_val, y_pred=y_pred, model_card="see confluence")),
    ("negative latency",
     dict(model=model, X=X_val, y_true=y_val, y_pred=y_pred, latencies_ms=[12.0, -3.0])),
]

logging.getLogger("bdp_model_gate").setLevel(logging.ERROR)
for label, kwargs in cases:
    try:
        ModelGate().run(StructuredGateContext(**kwargs))
        print(f"  (no error) {label}")
    except GateValidationError as exc:
        print(f"{label:28} -> {str(exc)[:78]}")
logging.getLogger("bdp_model_gate").setLevel(logging.INFO)

no model at all              -> no model supplied: pass either context.model (anything with .predict(), or a c
model without .predict()     -> context.model is a NotAModel, which has no .predict() method and is not callab
X is not a DataFrame         -> context.X must be a pandas DataFrame, got ndarray
y_true / X length mismatch   -> context.y_true has length 10, but context.X has 525 rows — they must be aligne


y_true has one class         -> context.y_true has only one unique value (array([1])) — most checks (AUC, disp


protected_df not aligned     -> context.protected_df has 5 rows, but context.X has 525 rows — they must be row


model_card not a dict        -> context.model_card must be a dict, got str
negative latency             -> context.latencies_ms contains negative values


## 9. Running it as a CI/CD gate

Installing the package provides a `bdp-model-gate` console script, meant as a
**pre-deployment step** — after training, before promotion. Not a per-PR
check.

| Exit | Status | Pipeline |
|---|---|---|
| `0` | `PASS` | deploy |
| `2` | `NEEDS_REVIEW` | pause for manual approval |
| `1` | `BLOCKED` | hard fail |

In [27]:
import subprocess
import sys
from pathlib import Path

import joblib

workdir = Path("cli_binary")
workdir.mkdir(exist_ok=True)

frame = X_val.copy()
frame["label"] = y_val
frame.to_csv(workdir / "validation.csv", index=False)
prot_val.to_csv(workdir / "protected.csv", index=False)
(workdir / "model_card.json").write_text(json.dumps(model_card, indent=2))
joblib.dump(model, workdir / "model.joblib")


def run_gate(*extra):
    proc = subprocess.run(
        [
            sys.executable, "-m", "bdp_model_gate.cli",
            "--model", str(workdir / "model.joblib"),
            "--data", str(workdir / "validation.csv"),
            "--target-col", "label",
            "--protected", str(workdir / "protected.csv"),
            "--model-card", str(workdir / "model_card.json"),
            "--task", "binary",
            "--cost-per-inference", "0.0009",
            "--output", str(workdir / "gate_report.json"),
            *extra,
        ],
        capture_output=True, text=True,
    )
    print(proc.stdout or proc.stderr)
    print("exit", proc.returncode, "->",
          {0: "PASS", 1: "BLOCKED", 2: "NEEDS_REVIEW"}.get(proc.returncode))
    return proc.returncode


run_gate("--metric", "roc_auc", "--min-score", "0.80")

Gate status: NEEDS_REVIEW (734ms, binary)
  roc_auc: 0.8636
  performance: 0 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 7 flag(s)
Full report written to cli_binary/gate_report.json

exit 2 -> NEEDS_REVIEW


2

`--config` accepts JSON, YAML or TOML, and CLI flags take precedence over the
file — so a pipeline can pin one threshold inline without a per-environment
config.

In [28]:
(workdir / "gate_config.yaml").write_text(
    """
performance:
  metric: roc_auc
  min_score: 0.60
fairness:
  disparity_threshold: 0.25
  proxy_corr_threshold: 0.99
security:
  adversarial_flip_rate_threshold: 0.40
""".lstrip()
)

print("--- lenient config ---")
run_gate("--config", str(workdir / "gate_config.yaml"))
print("\n--- same config, --min-score overrides it ---")
run_gate("--config", str(workdir / "gate_config.yaml"), "--min-score", "0.999")

--- lenient config ---


Gate status: NEEDS_REVIEW (714ms, binary)
  roc_auc: 0.8636
  performance: 0 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 6 flag(s)
Full report written to cli_binary/gate_report.json

exit 2 -> NEEDS_REVIEW

--- same config, --min-score overrides it ---


Gate status: BLOCKED (715ms, binary)
  roc_auc: 0.8636
  performance: 1 flag(s)
  compliance: 0 flag(s)
  security: 0 flag(s)
  fairness: 6 flag(s)
Full report written to cli_binary/gate_report.json

exit 1 -> BLOCKED


1

In GitHub Actions:

```yaml
- name: Model governance gate
  id: gate
  continue-on-error: true
  run: |
    bdp-model-gate --model model.joblib --data validation.csv \
      --target-col label --protected protected.csv \
      --model-card model_card.json --output gate_report.json

- name: Block on hard failure
  if: steps.gate.outcome == 'failure'
  run: exit 1

# exit code 2 -> route to an environment with required reviewers
```

Ready-to-adapt Azure Pipelines and GitHub Actions examples ship in
[`ci_examples/`](../ci_examples).

In [29]:
import shutil

shutil.rmtree(workdir, ignore_errors=True)
Path("gate_report.json").unlink(missing_ok=True)
print("cleaned up")

cleaned up


## 10. The report a reviewer reads

`NEEDS_REVIEW` hands the decision to a person, and everything above hands that
person JSON. `to_html()` writes one self-contained file instead: the verdict in
plain words, every finding with its evidence, and — for the checks whose
finding is a *shape* rather than a number — a chart beneath it.

No script, no stylesheet, no font, nothing fetched from anywhere. It opens
offline and prints clean, which is what a governance record has to do. Needs
the `[plots]` extra for the charts; without it the page renders text-only.

```bash
pip install "bdp-model-gate[plots]"
```

[Notebook 06](06_reports_and_plots.ipynb) covers the nine plots and how to
draw one for a check of your own.


In [30]:
from bdp_model_gate.plots import plotting_available

page = report.to_html("gate-report.html", title="Retail credit scorecard v4")
charts = page.count("<svg")
print(f"{len(page) / 1024:,.0f} KB, {charts} charts inlined "
      f"(plots installed: {plotting_available()})")

for forbidden in ("<script", "<link", "@import", 'src="http', 'href="http'):
    assert forbidden not in page, forbidden
print("nothing fetched from anywhere — open gate-report.html in any browser")

Path("gate-report.html").unlink(missing_ok=True)


413 KB, 5 charts inlined (plots installed: True)
nothing fetched from anywhere — open gate-report.html in any browser


## 11. Deprecations

Two renames still work but warn:

- `PerformanceConfig.min_accuracy` → `min_score` (plus `metric` to name what
  it applies to). The old name was genuinely misleading — it was compared
  against ROC AUC when scikit-learn was installed and accuracy when it was
  not, with nothing in the report saying which.
- `GateReport.model_auc` → `model_metric` + `model_score`, and it now returns
  `None` unless the metric really was AUC.

In [31]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    legacy = GateConfig()
    legacy.performance.min_accuracy = 0.85
    print("min_score is now:", legacy.performance.min_score)
    print("model_metric/model_score:", report.model_metric, report.model_score)
    print("model_auc:", report.model_auc)
    for w in caught:
        print(f"  {w.category.__name__}: {str(w.message)[:88]}")

min_score is now: 0.85
model_metric/model_score: roc_auc 0.8636
model_auc: 0.8636


## Summary

| Category | Checks | Blocking | Needs |
|---|---|---|---|
| Fairness | proxy correlation, disparate impact, SHAP subgroup, counterfactual, equalised odds, subgroup calibration | No → `NEEDS_REVIEW` | `protected_df` |
| Performance | score, calibration, p95 latency, cost | Yes | `y_true`/`y_pred` |
| Compliance | card fields, DPIA, explainability | Yes | `model_card` |
| Security | robustness, PII, prompt injection | Yes | optional `generate_fn` |

Three things worth carrying away:

1. **Optional inputs degrade, they don't fail.** Omit `protected_df` and
   fairness reports `NOT_APPLICABLE`; the gate grades what you give it.
2. **The metric is explicit.** `min_score` is meaningless without knowing
   what produced the score, so the report always names it and any fallback
   is loud.
3. **Fairness routes to a human.** It is the one category that does not
   block, because those flags need judgement.

Next: [02 multiclass and ordinal](02_multiclass_ordinal_sklearn.ipynb) ·
[03 regression](03_regression_sklearn.ipynb) ·
[04 any framework](04_any_framework_classification.ipynb) ·
[06 reports and plots](06_reports_and_plots.ipynb)